# 02 Model comparison and annotation agreement

Needs `data/labelled/labelled_pairs.csv` (your reviewed labelling sheet) and `data/raw/Hotel_Reviews.csv`.
**Colab:** use a GPU runtime if you can (Runtime > Change runtime type); the transformers also run on CPU, just slower. After changing anything in `src/`, use Runtime > Restart session.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/hotel-review-aspect-sentiment/notebooks')
except ImportError:
    pass

In [ ]:
# Colab usually has transformers/torch already. Uncomment if an import below fails:
# !pip install -q transformers sentencepiece
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.pipeline import load_reviews, split_sentences, add_difficulty_flags
from src import models as M
from src import evaluate as E

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "Hotel_Reviews.csv"
LABELS_PATH = ROOT / "data" / "labelled" / "labelled_pairs.csv"
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
SEED, N_REVIEWS = 42, 20000      # must match notebook 01

## 1. Load your labels and check them
Stops with a clear message if any row has a blank or misspelt `gold_sentiment` / `difficulty`.

In [ ]:
lab = E.load_labelled(LABELS_PATH)
print(len(lab), "labelled pairs")
print(lab["gold_sentiment"].value_counts())
print(lab["difficulty"].value_counts())

## 2. Annotation agreement: your labels vs the AI pre-labels
The AI labels were a starting point that you reviewed and corrected, so this is **not** independent inter-annotator agreement. It shows how often the AI draft needed fixing, and where. Cohen's kappa corrects raw agreement for chance. Report both, and say plainly in the report that the labels are AI-assisted and human-verified.

In [ ]:
ag = E.annotation_agreement(lab)
for k, v in ag.items():
    if k not in ("confusion", "changed_rows"):
        print(f"{k:24s}", round(v, 3) if isinstance(v, float) else v)
ag["confusion"]

In [ ]:
# Every row where you disagreed with the AI draft (good material for the report's annotation section).
ag["changed_rows"].to_csv(OUT / "annotation_changes.csv", index=False)
ag["changed_rows"]

## 3. Rebuild the sentences and the weak-label training data
Same sample as notebook 01 (same seed). Training data for the TF-IDF model = all comments of at least 3 words whose **review** is not among the labelled ones, so nothing you hand-labelled leaks into training.

In [ ]:
df = load_reviews(DATA_PATH, n_sample=N_REVIEWS, seed=SEED)
flagged = add_difficulty_flags(split_sentences(df))

key = lambda r, s: (r, s)
field_by_key = {(r, s): f for r, s, f in zip(flagged["review_id"], flagged["sent_id"], flagged["field"])}
missing = [k for k in zip(lab["review_id"], lab["sent_id"]) if k not in field_by_key]
assert not missing, f"{len(missing)} labelled sentences not found: N_REVIEWS/SEED differ from notebook 01?"

train = flagged[(flagged["n_words"] >= 3) & (~flagged["review_id"].isin(lab["review_id"]))]
print(len(train), "weak-labelled training comments")

## 4. Fit and run the models
Only rows with a real sentiment are scored (`not_applicable` rows are keyword false alarms, so they are excluded from model scoring and reported separately).

In [ ]:
ev = E.evaluable(lab)
print(len(ev), "rows scored;", len(lab) - len(ev), "not_applicable rows excluded")

preds = {}
majority = M.MajorityModel().fit(train["field"])
preds[majority.name] = majority.predict(ev)
preds[M.BoxLabelModel(field_by_key).name] = M.BoxLabelModel(field_by_key).predict(ev)
tfidf = M.TfidfLRModel(SEED).fit(train["sentence"], train["field"])
preds[tfidf.name] = tfidf.predict(ev)
{k: pd.Series(v).value_counts().to_dict() for k, v in preds.items()}

In [ ]:
# Pretrained transformers (download on first run). Wrapped so one failure does not stop the notebook.
for cls in (M.HFSentimentModel, M.HFAbsaModel):
    try:
        m = cls(device=0 if __import__("torch").cuda.is_available() else -1)
        preds[m.name] = m.predict(ev)
        print("ok:", m.name)
    except Exception as e:
        print("FAILED:", cls.__name__, "->", type(e).__name__, str(e)[:200])

## 5. Results by difficulty group
The central question: do the models cope with negation, contrast and several aspects in one comment? Only the ABSA model is told *which aspect* to judge, so on multi-aspect and contrast rows it has a real advantage on paper. Check whether the numbers show it.

In [ ]:
scores = E.score_models(ev, preds)
table = scores.pivot(index="model", columns="subset", values="macro_f1")[["all"] + [d for d in E.DIFFICULTIES if d in set(scores["subset"])]].round(3)
scores.to_csv(OUT / "model_scores.csv", index=False)
table

In [ ]:
fig = E.plot_by_difficulty(scores, "accuracy", OUT / "accuracy_by_difficulty.png")
plt.show()

In [ ]:
# With about 200 rows (and fewer per group) differences are noisy: 95% bootstrap intervals for overall accuracy.
ci = {m: E.bootstrap_ci(ev, p, "accuracy", seed=SEED) for m, p in preds.items()}
pd.DataFrame(ci, index=["low", "high"]).T.round(3)

## 6. Error analysis
Read these. The report's discussion should quote a few real failures per hard case and explain why they happen.

In [ ]:
errors = pd.concat([E.error_table(ev, p, m) for m, p in preds.items()
                    if m not in ("Majority class", "Box label (weak reference)")])
errors.to_csv(OUT / "model_errors.csv", index=False)
pd.set_option("display.max_colwidth", 140)
errors.groupby(["model", "difficulty"]).size().unstack(fill_value=0)

In [ ]:
# Rows where the aspect-aware model is right and the aspect-blind RoBERTa is wrong (and the reverse).
a, b = "DeBERTa ABSA (pretrained)", "RoBERTa sentiment (pretrained)"
if a in preds and b in preds:
    y = ev["gold_sentiment"]
    pa, pb = pd.Series(preds[a]), pd.Series(preds[b])
    print("ABSA right, RoBERTa wrong:", int(((pa == y) & (pb != y)).sum()),
          "| RoBERTa right, ABSA wrong:", int(((pb == y) & (pa != y)).sum()))
    ev.assign(absa=pa, roberta=pb)[(pa == y) & (pb != y)][["difficulty", "aspect", "sentence", "gold_sentiment", "absa", "roberta"]].head(15)

## 7. Optional: an LLM prompt
Supply any function that sends a prompt to an LLM and returns its text. **Circularity caveat:** the pre-labels came from an LLM, so an LLM scoring well here is partly agreeing with its own family's draft, even after your corrections. Say so in the report if you use this.

In [ ]:
# Example (needs your own API key; leave commented if you skip this):
# ask = lambda p: your_llm_call(p)  # any function that takes a prompt and returns text
# llm = M.LLMPromptModel(ask, "LLM prompt"); preds[llm.name] = llm.predict(ev)

## Next steps
1. Put the table, chart and 3 to 5 real error examples in report section 3 (Results and discussion).
2. Mention the annotation process and its limits (AI-assisted, one human verifier, about 200 rows) in section 2.
3. Build the Gradio demo around the best model.